<figure>
  <IMG SRC="https://raw.githubusercontent.com/fhswf/datasets/refs/heads/main/images/fh-swf-logo.svg" WIDTH=250 ALIGN="right">
</figure>

# Programmierung für KI
### Winterersemester 2026/27
Prof. Dr. Heiner Giefers

**Achtung:** Im folgenden Aufgabenblatt sind einige Funktionen enthalten, die Sie zur Lösung der Aufgaben *verwenden* müssen. Sie müssen die Implementierung der Funktionen nicht unbedingt *verstehen*, um sie zu verwenden. Um möglichst viele Aufgaben zu lösen, verwenden Sie die Funktionen einfach "wie sie sind" und schauen Sie sich die Details im Nachgang an.

# Sprache von Texten erkennen

In dieser Aufgabe wollen wir uns mit dem Problem beschäftigen, die Sprache zu bestimmen, in der ein gewisser Text verfasst ist.
Es gibt verschiedenste Ansätze, dieses Problem zu lösen.
Wir werden ein sehr einfaches, Daten-getriebenes Verfahren einsetzen, bei dem die charakteristische *Buchstabenhäufigkeit* einer Sprache verwendet wird.

Die Idee ist folgende: In den verschiedenen Sprachen kommen bestimmte Buchstaben unterschiedlich häufig vor.
Im Englischen, zum Beispiel, ist das *y* deutlich häufiger als im Deutschen, im Französischen ist das *e* (ohne Akzent) wahrscheinlich seltener als in anderen Sprachen.

Wenn wir bestimmen, wie häufig die einzelnen Buchstaben in einer Sprache *im Allgemeinen* vorkommen, haben wir eine Art *Fingerabdruck* der Sprache. Für einen neuen Text, zu dem wir die Sprache nicht kennen, können wir ebenfalls die Buchstabenhäufigkeit berechnen und anschließend diese Verteilung mit allen Sprach-Fingerabdrücken vergleichen. Unsere Annahme ist, dass der Fingerabdruck, zu dem unsere Buchstaben-Verteilung am besten passt, zur Sprache gehört, in der auch der Text verfasst ist.

Im Folgenden wollen wir versuchen herauszufinden, ob diese Idee funktioniert. Dazu müssen wir folgende Teilprobleme lösen:

1. Wir müssen **Texte in verschiedenen Sprachen beschaffen**. Diese Texte müssen jeweils möglichst repräsentativ für die Sprache sein. Fachtexte oder andere *spezielle* Texte sollten eher vermieden werden.
1. Wir müssen in den Texten die Buchstaben analysieren und die **relative Häufigkeit jedes Buchstaben bestimmen**. Dieser Schritt muss für alle Sprachen wiederholt werden.
1. Um das Verfahren zu testen, müssen wir neue, diesmal kürzere Textpassagen beschaffen. Wir können die gleichen Funktionen wie im Schritt zuvor verwenden, um die Buchstabenhäufigkeit zu bestimmen.
1. Wir benötigen einen Weg um herauszufinden, wie *ähnlich* zwei verschiedene Häufigkeitsverteilungen sind. Wir werden hierfür die **Abweichung zwischen den Verteilungen** bestimmen und das Minimum dieser Abweichungen als Ergebnis auswählen.

### Schritt 1: Texte in verschiedenen Sprachen beschaffen
Um die mittleren Buchstabenhäufigkeiten in den verschiedenen Sprachen zu ermitteln, brauchen wir zunächst einmal **Daten**.
Eine recht gute Quelle für Texte in verschiedenen Sprachen ist [Wikipedia](https://www.wikipedia.org).
Wir laden zufällige Artikel direkt über die MediaWiki-API herunter. Dafür genügen `urllib` und `json` aus der Standardbibliothek; zusätzliche Pakete sind nicht nötig.

Wikipedia verlangt für automatisierte Zugriffe einen beschreibenden User-Agent mit Kontaktmöglichkeit. In diesem Aufgabenblatt verwenden wir `verbund-aki@fh-swf.de`.

In [ ]:
import json
import re
import unicodedata
from urllib.parse import urlencode
from urllib.request import Request, urlopen

Die nächste Code-Zelle definiert `random_page` und `random_text`. `random_text(size, lang)` liefert einen bereinigten Text mit genau `size` Zeichen.

`random_page` fragt die MediaWiki-API nach einem zufälligen Artikel samt Klartext-Auszug. Beide Funktionen verwenden dieselbe Textbereinigung, die wir auch für die Testpassage nutzen.

In [ ]:
USER_AGENT = "SprachenErkennen/1.0 (mailto:verbund-aki@fh-swf.de)"

def clean_text(text):
    text = text.lower()
    text = text.translate(str.maketrans({
        "ä": "ae", "ö": "oe", "ü": "ue", "ß": "ss"
    }))
    text = unicodedata.normalize("NFKD", text)
    text = "".join(char for char in text if not unicodedata.combining(char))
    text = re.sub(r"[^a-z ]", " ", text)
    return " ".join(text.split())


def get_wiki_text(lang, params):
    params.update({
        "action": "query",
        "prop": "extracts",
        "explaintext": 1,
        "format": "json",
        "formatversion": 2,
    })
    url = f"https://{lang}.wikipedia.org/w/api.php?{urlencode(params)}"
    request = Request(url, headers={"User-Agent": USER_AGENT})
    # urlopen löst bei HTTP-Fehlern (z. B. 404) automatisch eine Exception aus
    with urlopen(request, timeout=10) as response:
        data = json.load(response)
    pages = data.get("query", {}).get("pages", [])
    if not pages or not pages[0].get("extract"):
        raise ValueError("Die Wikipedia-API hat keinen Artikeltext geliefert")
    return pages[0]["extract"]


def random_page(lang="de"):
    return get_wiki_text(lang, {"generator": "random", "grnnamespace": 0})

def get_wiki_page(term, lang="de"):
    text = get_wiki_text(lang, {"titles": term})
    return clean_text(text)



def random_text(size, lang="de"):
    if size < 0:
        raise ValueError("size darf nicht negativ sein")
    text = ""
    while len(text) < size:
        text += clean_text(random_page(lang)) + " "
    return text[:size]


Wir können nun die Funktion `random_text` verwenden, um einen String mit beliebiger Länge in verschiedenen Sprachen zu erzeugen.
Im folgenden Beispiel erzeigen wir einen String `de_text_500` aus dem Deutschen mit genau 500 Zeichen

In [ ]:
de_text_500 = random_text(500,"de")
print(de_text_500)

Zeichenweise durchlaufen, können wir diesen String wie jede Zeichenkette, ganz einfach mit `for`:

In [ ]:
for z in de_text_500:
    #Drucke jedes Zeichen und danach jeweils ein '-'
    print(z,end='-')

### Schritt 2: Relative Häufigkeiten der Buchstaben bestimmen

Wir möchten nun die relative Häufigkeit der Buchstaben in den Texten bestimmen. Dazu müssen wir zunächst einmal die Vorkommen der Buchstaben zählen. Wir werden uns auf die 26 Buchstaben des Alphabets beschränken und alle anderen Zeichen ignorieren. Außerdem werden wir Groß- und Kleinschreibung nicht unterscheiden.

Für das Zählen von Objekten kann man in Python Dictionaries verwenden. Für jedes Objekt (in unserem Fall jeden Buchstaben) wird ein Schlüssel im Dictionary angelegt, der auf die Anzahl der Vorkommen des Objekts verweist. Wenn ein Objekt noch nicht im Dictionary enthalten ist, kann man es mit einem Standardwert initialisieren.

**Aufgabe:** Bestimmen Sie die relativen Häufigkeiten der Buchstaben im Text.

Dazu verwenden wir ein *Dictionary* `buchstaben`, das zu Anfang leer ist und gehen folgendermaßen vor:
1. Wir durchlaufen alle Buchstaben des bereinigten Textes. Leerzeichen und andere Zeichen werden nicht gezählt.
2. Wir prüfen, ob der aktuelle Buchstabe als *key* im Dictionary vorkommt
    1. Falls er nicht vorkommt, fügen wir den Buchstaben als Schlüssel hinzu und setzen den Wert des Schlüssels auf 1.
    2. Falls er vorkommt, erhöhen wir den Wert des Eintrags um 1
3. Nachdem alle Buchstaben verarbeitet sind, teilen wir jeden Wert durch die Anzahl der Buchstaben im Text. Beispiel: Wenn der Text 1000 Buchstaben hat und `buchstaben['e']` gleich 100 ist, so setzen wir `buchstaben['e']` auf 0.1.

In [ ]:
def letter_frequency(text):
    buchstaben = {}
    # YOUR CODE HERE
    raise NotImplementedError()
    
    return buchstaben


In [ ]:
# Einfache Tests ohne Wikipedia-Zugriff
assert clean_text("ÄRGER, École und Straße!") == "aerger ecole und strasse"
assert letter_frequency("a a") == {"a": 1.0}

Wir haben nun eine Methode `letter_frequency`, mit der wir die relativen Häufigkeiten der Buchstaben eines Textes bestimmen können.
Um einen ersten Eindruck von diesen Häufigkeiten zu bekommen, werden wir ein Säulendiagramm erstellen.
Dazu verwenden wir die Bibliothek `matplotlib`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

text = random_text(1000, 'de')
buchstaben = letter_frequency(text)

plt.figure(figsize=(30,5)) 

x = range(len(buchstaben))
y = list(buchstaben.values())
x_ticks = list(buchstaben.keys())
plt.bar(x, y, align='center')
plt.xticks(x, x_ticks, rotation=0, fontsize=18)
plt.show()

Wie man sieht, sind die Einträge im Dictionary unsortiert.
Weder die Schlüssel noch die Werte folgen einer Ordnung.

Um die Buchstabenhäufigkeiten aber später vergleichen zu können, wollen wir einen *Vektor* erzeugen, bei dem Buchstaben immer an gleicher Stelle stehen. Z.B. `a` an Index 0, `b` an Index 1, usw.
Eine Idee könnte sein, die Schlüssel zu sortieren.
Hierbei ist aber Vorsicht geboten, denn es könnt sein, dass ein Buchstabe im Text nicht vorkommt.
In dem Fall, hätte der Vektor zu wenige Einträge.

Wir verfolgen daher einen anderen Weg.
Über eine `for`-Schleife zählen wir alle Buchstaben von *a* bis *z* auf, schauen nach, ob sie im Dictionary enthalten sind und übernehmen den Wert in eine Liste.
Ist der Buchstabe nicht enthalten, übernehmen wir den Wert 0.0 in die Liste.

Eine Schleife, die alle Buchstaben durchläuft könnte so aussehen
```python
for i in range(ord('a'), ord('z')+1):
    print(chr(i), end=' ')
```

`ord('a')` liefert den ASCII Code des Zeichens *a*.
So kann die Schleife die ASCII Codes von *a* bis *z* durchlaufen.
Im Schleifenrumpf erzeugen wir aus dem ASCII Code mit der Typumwandlung `chr` wieder ein Zeichen.

**Aufgabe:** Schreiben Sie eine Funktion `frequency_vector(d)`, die aus einem Dictionary `d` ein Tupel mit 26 Einträgen erzeugt. Die Einträge enthalten die relativen Häufigkeiten der Buchstaben von *a* bis *z*.

In [ ]:
def frequency_vector(d):
    f = []
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
# Test Zelle
test_d = {'a': 0.5, 'd': 0.5}
test_l = [0.0]*26
test_l[0]=0.5
test_l[3]=0.5
assert frequency_vector(test_d)==tuple(test_l)

In der folgenden Code-Zelle werden die zuvor entwickelten Funktionen verwendet, um die relativen Häufigkeiten der Buchstaben in den Sprachen *Deutsch*, *Englisch*, *Spanisch* und *Französisch* zu bestimmen.
Wir verwenden dazu jeweils 10000 Zeichen aus Wikipedia Artikeln.
Die Resultate werden im Dictionary `haeufigkeiten` unter den Schlüsseln `'de'`, `'en'`, `'es'` und `'fr'` abgelegt.

In [ ]:
languages = ['de', 'en', 'es', 'fr']

def get_lang_freq(lang):
    text = random_text(10000, lang)
    buchstaben = letter_frequency(text)
    return frequency_vector(buchstaben)

haeufigkeiten = {}
for l in languages:
    haeufigkeiten[l] = get_lang_freq(l)

### Schritt 3: Textpassage in einer Sprache beschaffen

Nun haben wir verschiedene Vektoren, die wir als Maßstab für die einzelnen Sprachen verwenden können.
Als Nächstes wollen wir überprüfen, ob wir diese Vektoren verwenden können, um für einen neuen Text vorherzusagen, in welcher Sprache er geschrieben ist.

Dazu definiert die folgende Code-Zelle die Funktionen `get_wiki_page` und `get_page_freq`. Sie laden den Klartext einer bestimmten Wikipedia-Seite, bereinigen ihn auf dieselbe Weise wie die Trainingsdaten und berechnen seine relative Zeichenhäufigkeit.

In [ ]:
def get_page_freq(term, lang='de'):
    text = get_wiki_page(term, lang=lang)
    buchstaben = letter_frequency(text)
    return frequency_vector(buchstaben)

mypage = get_page_freq("Hochschule")

### Schritt 4: Häufigkeiten vergleichen (Fehler bestimmen)

An dieser Stelle haben wir vier Vektoren, je einen für *Deutsch*, *Englisch*, *Spanisch* und *Französisch*, sowie einen weiteren für *unseren* Text. Die Vektoren enthalten die relativen Buchstabenhäufigkeiten.
Nun wollen wir ausrechnen, zu welchem der vier Sprach-Vektoren unser Buchstaben-Vektor *am ähnlichsten* ist, bzw. zu welchem Sprach-Vektor er die wenigsten Unterschiede aufweist.

Wir werden also die einzelnen Buchstaben-Häufigkeiten in unserem Vektor mit denen im Sprach-Vektor *vergleichen*.
Mathematisch gesehen, bilden wir die Differenz der Häufigkeiten aller einzelnen Buchstaben.
Kommt z.B. in unserem Text das *e* zu 10% und im Sprach-Vektor zu 8% vor, so ist die Abweichung $0.1-0.08=0.02$ für den Buchstaben *e*.

Die folgende Abbildung zeigt ein Beispiel anhand zweier Sprachen und der Buchstaben *a, e, i, o, u*.
Vergleicht man die Häufigkeiten der Sprachen *1* und *2* jeweils mit der Häufigkeit der Zeichen im Text, so ergibt sich für jedes Zeichen ein Fehler. Ein kleinerer Fehler bedeutet, dass der Text dieser Sprache ähnlicher ist. Das Verfahren berechnet keine statistischen Wahrscheinlichkeiten.

![](https://github.com/fhswf/datasets/raw/main/images/letterfreq.png)

Wenn wir nun alle Abweichungen aufsummieren, kann es zu einem Problem kommen.
Manche Differenzen werden positiv, andere negativ sein.
Eine einfache Summe wird die negativen und positiven Abweichungen gegeneinander aufrechen.

Um dieses Problem zu beheben, gibt es verschiedene Möglichkeiten.
Man kann die Absolutwerte der Abweichungen aufsummieren, oder auch deren Quadratwerte.
In jedem Fall wird die Summe am Ende durch die Anzahl der Einträge des Vektors geteilt, um den Mittleren Fehler zu berechnen.

Verwendet man zum Berechnung die Absolutwerte, nennt man das Maß [*Mittlerer absoluter Fehler*](https://de.wikipedia.org/wiki/Mittlerer_absoluter_Fehler) (oder *mean absolute error*), verwendet man die Fehlerquadrate heißt das Maß [*Mittlerer quadratischer Fehler*](https://de.wikipedia.org/wiki/Mittlere_quadratische_Abweichung) (oder *mean squared error*).

**Aufgabe:** Implementieren Sie die Funktion `mse` zur Berechnung des Mittleren Quadratischen Fehlers. 

*Hinweise:*
- Die Funktion soll zwei Listen oder Tupel `a` und `b` gleicher Länge als Parameter übergeben bekommen.
- Die Funktion berechnet die Differenzen der Werte an gleichen Index-Positionen.
- Da diese Differenten positiv oder negativ sein können, bilden wir jeweils das Quadrat und summieren diese quadrierten Fehler-Werte auf.
- Am Ende soll die Funktion den Mittleren quadratischen Fehler mit `return` zurückgeben.

*Beispiel:* Für die Beiden Listen `x` und `y` ist der mittlere quadratische Fehler gleich 2:
```
x = [1,2,3,5]
y = [3,2,5,5]
```

$\text{MSE} = \frac{(1-3)^2 + (2-2)^2 + (3-5)^2 + (5-5)^2}{4} = \frac{(-2)^2 + 0^2 + (-2)^2 + 0^2}{4} = \frac{4 + 0 + 4 + 0}{4} = \frac{8}{4} = 2$ 


In [ ]:
def mse(a,b):
    # YOUR CODE HERE
    raise NotImplementedError()

# Liefere eine Liste geordnet nach dem mittleren quadratischen Fehler (MSE) der Vektoren
def get_best_language(mypage, languages):
    fehler = {}
    for lang in languages:
        fehler[lang] = mse(mypage, languages[lang])
    res = ""
    for lang in sorted(fehler, key=fehler.get):
        res += f"{lang}: {fehler[lang]:.4f}\n"
    return res

# Deterministische Tests mit festen Vektoren
a = [1, 2, 3, 4, 5]
b = [3, 2, 5, 4, 5]
assert mse(a, b) == 1.6
test_page = (0.5, 0.5)
test_languages = {"de": (0.4, 0.6), "en": (0.1, 0.9)}
assert get_best_language(test_page, test_languages) == "de: 0.0100\nen: 0.1600\n"

In [ ]:
# Test Zelle
a = [1,2,3,4,5]
b = [3,2,5,4,5]
assert mse(a,b)==1.6

Nun können wir unsere Vektoren vergleichen und die Sprache mit dem kleinsten mittleren quadratischen Fehler auswählen.

In [ ]:
best_language = get_best_language(mypage, haeufigkeiten)
print("Ähnlichste Sprache:")
print(best_language)

`best_language` enthält den Sprachcode mit dem kleinsten mittleren quadratischen Fehler. Das Ergebnis ist eine einfache Zuordnung anhand von Zeichenhäufigkeiten, keine statistische Wahrscheinlichkeit. Bei einer sehr kurzen Testpassage kann die Zuordnung ungenau sein.

### Ausblick: n-Gramme statt Buchstaben verwenden

Wie man sieht, sind Buchstabenhäufigkeiten ein mögliches Vorhersagekriterium, aber allein oft nicht ausreichend. Deutsch und Englisch können ähnliche Häufigkeitsmuster haben und deshalb leicht verwechselt werden.

Um das Verfahren zu verbessern, könnte man statt einzelne Buchstaben ganze Worte betrachten.
Oder alternativ dazu, Sequenzen von $n$ aufeinanderfolgenden Buchstaben, den sogenannten [n-Grammen](https://de.wikipedia.org/wiki/N-Gramm).

Wenn man einen Text auf n-Gramme untersucht, betrachtet man Folgen von $n$ Buchstaben von jedem Buchstaben aus.
In der folgenden Code-Zelle verwenden wir das Wort *Waschmaschine* als Beispiel.
Das Wort hat 13 Buchstaben und besitzt damit 11 Trigramme (3-Gramme) von denen eines (das *sch*) doppelt vorkommt.

In [ ]:
wort = 'waschmaschine'
[wort[i:i+3] for i in range(len(wort)-2)]

**Aufgabe:** 
1. Generieren Sie einen zufälligen Text mit 2000 Zeichen
2. Teilen Sie den Text so auf, dass Sie die einzelnen Worte in einer Liste ablegen. Berechnen Sie die 3-Gramme für alle Worte im Text und speichern Sie die 3-Gramme in einer Liste
3. Es sollen nur die 3-Gramme betrachtet werden. D.h, wenn ein Wort weniger als 3 Buchstaben besitzt, nehmen Sie es nicht in die Liste auf
4. Geben Sie die Anzahl aller 3-Gramme aus, die in ihrem Text enthalten sind
5. Geben Sie die Anzahl aller eindeutigen 3-Gramme aus, d.h. ohne doppelte Vorkommen

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()